# Capstone RAG — local LLM on the Kaggle GPU

Retrieval over a Qdrant collection, with the answer generated by a Qwen2.5
model **running on this session's GPU** rather than the Hugging Face Inference
API.

**Session options must be set before running anything:**

| Setting | Value | Why |
|---|---|---|
| Accelerator | **GPU T4 x2** | 14B-AWQ needs both cards. P100 is compute capability 6.0 and has no AWQ kernels. |
| Internet | **On** | Weight downloads and the Qdrant round trip. |

Run the cells in order. The order matters: the embedding model is loaded and
then *freed* before vLLM starts, because vLLM claims a fixed slice of VRAM at
startup and cannot give it back.

## 1. Install

In [ ]:
# vLLM pins its own torch build, so install it first and let the rest resolve
# around it. This takes several minutes on a cold session.
!pip install -q vllm openai

!pip install -q llama-index-core llama-index-vector-stores-qdrant \
                llama-index-embeddings-huggingface \
                qdrant-client python-dotenv streamlit pyngrok

## 2. Configuration

Every credential and model name lives **here and only here**. The previous
version of this notebook declared them separately in the embedding cell and the
`app.py` cell, which is how those two ended up pointing at two different Qdrant
clusters. Everything downstream reads `os.environ`, and the Streamlit and vLLM
subprocesses inherit it.

Set these in **Add-ons → Secrets** in the Kaggle menu. Nothing is hardcoded, so
this notebook is safe to commit.

In [ ]:
import os


def secret(name, default=None, required=True):
    """Read a credential from Kaggle Secrets, falling back to the environment.

    Kaggle Secrets is the only one of the two that exists on a fresh session;
    the environment fallback is what makes the same notebook runnable locally.
    """
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except Exception:
        pass

    value = os.environ.get(name, default)
    if required and not value:
        raise RuntimeError(
            f"{name} is not set. Add it under Add-ons -> Secrets, "
            f"or export it in the environment."
        )
    return value


# --- Qdrant -------------------------------------------------------------
os.environ["QDRANT_URL"] = secret("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = secret("QDRANT_API_KEY")
os.environ["QDRANT_COLLECTION"] = "capstone"

# --- Embeddings ---------------------------------------------------------
# 1024-dim and multilingual, which the Indonesian source documents need.
# Changing this invalidates every stored vector: a query embedded with a
# different model still has the right dimensionality, so Qdrant accepts it
# and silently returns noise. Re-run the embedding step after any change.
os.environ["EMBED_MODEL"] = "BAAI/bge-m3"

# --- Generation ---------------------------------------------------------
# The AWQ 4-bit build is ~10 GB and fits across two T4s with room for the KV
# cache; the fp16 14B is ~28 GB and would not.
os.environ["LLM_BACKEND"] = "local"
os.environ["HF_MODEL"] = "Qwen/Qwen2.5-14B-Instruct-AWQ"
os.environ["LOCAL_LLM_URL"] = "http://localhost:8000/v1"

# --- Tunnel -------------------------------------------------------------
os.environ["NGROK_AUTHTOKEN"] = secret("NGROK_AUTHTOKEN")

# --- Disk ---------------------------------------------------------------
# Kaggle's root filesystem is small and the default HF cache lives there, so a
# 10 GB download dies partway through. /kaggle/working has room.
os.environ["HF_HOME"] = "/kaggle/working/hf"

print("Collection:", os.environ["QDRANT_COLLECTION"])
print("Embeddings:", os.environ["EMBED_MODEL"])
print("Generation:", os.environ["HF_MODEL"], "(local vLLM)")

## 3. Build the Qdrant index

This runs **before** vLLM starts, so the embedding model has the whole GPU to
itself and the ~2,200 chunks embed quickly.

Skip this cell if the collection is already populated and the embedding model
has not changed.

In [ ]:
import os
import re
import json
from pathlib import Path

from dotenv import load_dotenv
from llama_index.core import Document, Settings, StorageContext, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import TextNode
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import qdrant_client

load_dotenv()

qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")
qdrant_collection = os.getenv("QDRANT_COLLECTION", "bni_training")

# Must match app.py. Both read the same env var so the two cannot drift apart:
# a query embedded with a different model than the stored vectors still has the
# right dimensionality, so Qdrant accepts it and silently returns noise.
EMBED_MODEL_NAME = os.getenv("EMBED_MODEL", "BAAI/bge-m3")  # 1024-dim, multilingual

# The collection holds vectors from one model only. Set RECREATE_COLLECTION=0 to
# append instead, but changing EMBED_MODEL without a wipe leaves the collection
# holding two incompatible vector spaces at once.
RECREATE_COLLECTION = os.getenv("RECREATE_COLLECTION", "1") not in ("0", "false", "False")

Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)

client = qdrant_client.QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key,
    # The cluster is in sa-east-1; the 5s default is not enough for a
    # round trip with 1024-dim vectors and the writes time out.
    timeout=120,
    check_compatibility=False,
)
if RECREATE_COLLECTION and client.collection_exists(qdrant_collection):
    print(f"Deleting existing collection {qdrant_collection} so it is rebuilt with {EMBED_MODEL_NAME}.")
    client.delete_collection(qdrant_collection)

vector_store = QdrantVectorStore(
    client=client,
    collection_name=qdrant_collection,
    batch_size=8,
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)


# --------------------------------------------------------------------------
# Source 1: PDF-derived JSON (pdfs/parsed_result and data/parsed_result)
# --------------------------------------------------------------------------

def _page_documents(json_path, pages):
    """parse.py output: a list of page objects with text/table blocks."""
    documents = []

    for page in pages:
        content = [block["text"] for block in page.get("text_blocks", [])]
        content.extend(
            "\n".join(" | ".join(row) for row in table["data"])
            for table in page.get("table_blocks", [])
        )

        if content:
            documents.append(Document(
                text="\n".join(content),
                metadata={
                    "source": json_path.name,
                    "page": page["page"],
                },
            ))

    return documents


def _flat_document(json_path, payload):
    """Flat {"text": "..."} output -- one blob, no page structure to keep."""
    text = payload.get("text", "").strip()
    if not text:
        return []

    return [Document(text=text, metadata={"source": json_path.name})]


def _purchase_request_document(json_path, payload):
    """parse_model.py output: a structured PurchaseRequest object.

    Rendered as readable "key: value" prose so the embedding model has real
    language to work with rather than raw JSON punctuation.
    """
    search = payload.get("search_summary") or {}

    lines = [
        f"Purchase request {payload.get('request_id', 'unknown')} "
        f"from {payload.get('department', 'unknown department')}.",
        f"Item category: {payload.get('item_category')}.",
        f"Specs searched: {search.get('specs_searched')}.",
        f"Quantity: {payload.get('quantity')} unit(s).",
        f"Requested unit price: {payload.get('requested_unit_price')} IDR.",
        f"Historical average price: {payload.get('historical_avg_price')} IDR.",
        f"Price variance ratio: {payload.get('price_variance_ratio')}.",
        f"Total amount: {payload.get('total_amount')} IDR.",
        f"Cheapest vendor found: {search.get('cheapest_vendor_found')} "
        f"via {search.get('vendor_channel_type')}.",
        f"Real price found: {search.get('real_price')} IDR "
        f"(saving {search.get('price_savings_vs_requested')} IDR vs requested).",
        f"Vendor risk score: {payload.get('vendor_risk_score')}.",
        f"Department budget remaining: {payload.get('dept_budget_remaining')} IDR.",
        f"Urgent: {'yes' if payload.get('is_urgent') else 'no'}.",
    ]

    return [Document(
        text="\n".join(lines),
        metadata={
            "source": json_path.name,
            "doc_type": "purchase_request",
            "request_id": payload.get("request_id"),
            "department": payload.get("department"),
        },
    )]


def load_parsed_documents(json_paths):
    documents = []

    for json_path in json_paths:
        try:
            payload = json.loads(json_path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, UnicodeDecodeError) as e:
            print(f"  {json_path.name}: skipped, unreadable ({e})")
            continue

        if isinstance(payload, list):
            found = _page_documents(json_path, payload)
        elif isinstance(payload, dict) and "text" in payload:
            found = _flat_document(json_path, payload)
        elif isinstance(payload, dict) and "request_id" in payload:
            found = _purchase_request_document(json_path, payload)
        else:
            print(f"  {json_path.name}: skipped, unrecognised shape ({type(payload).__name__})")
            continue

        print(f"  {json_path.name}: {len(found)} document(s)")
        documents.extend(found)

    return documents


# --------------------------------------------------------------------------
# Source 2: laptop listings (data/laptop_listings_filled.json)
# --------------------------------------------------------------------------

def _listing_text(listing):
    """Render one marketplace listing as prose for embedding."""
    spec_bits = []
    for label, key, unit in [
        ("CPU", "cpu", ""),
        ("RAM", "ram_gb", " GB"),
        ("Storage", "storage_gb", " GB"),
        ("GPU", "gpu", ""),
        ("Screen", "screen_size_in", " inch"),
    ]:
        value = listing.get(key)
        if value is not None:
            spec_bits.append(f"{label}: {value}{unit}")

    lines = [
        listing.get("title") or "Untitled listing",
        f"Brand: {listing.get('brand')}. Model: {listing.get('model')}.",
    ]
    if spec_bits:
        lines.append("Specifications -- " + ", ".join(spec_bits) + ".")

    lines.append(
        f"Price: {listing.get('price_idr')} IDR"
        + (f" (original {listing['original_price_idr']} IDR)"
           if listing.get("original_price_idr") is not None else "")
        + f". Condition: {listing.get('condition')}."
    )
    lines.append(
        f"Sold by {listing.get('seller_name')} on {listing.get('source')} "
        f"({'official store' if listing.get('seller_is_official') else 'regular seller'}), "
        f"located in {listing.get('location')}."
    )

    if listing.get("seller_rating") is not None:
        lines.append(
            f"Seller rating: {listing['seller_rating']}"
            + (f" from {listing['seller_num_reviews']} reviews"
               if listing.get("seller_num_reviews") is not None else "")
            + "."
        )

    if listing.get("is_suspected_scam"):
        reasons = ", ".join(listing.get("scam_reasons") or []) or "unspecified"
        lines.append(f"WARNING: flagged as a suspected scam listing. Reasons: {reasons}.")
    else:
        lines.append("Not flagged as a suspected scam listing.")

    return "\n".join(lines)


def load_listing_documents(listings_path):
    listings = json.loads(Path(listings_path).read_text(encoding="utf-8"))

    documents = [
        Document(
            text=_listing_text(listing),
            metadata={
                "source": Path(listings_path).name,
                "doc_type": "laptop_listing",
                "marketplace": listing.get("source"),
                "source_id": listing.get("source_id"),
                "brand": listing.get("brand"),
                "price_idr": listing.get("price_idr"),
                "is_suspected_scam": bool(listing.get("is_suspected_scam")),
                "url": listing.get("url"),
            },
        )
        for listing in listings
    ]

    print(f"  {Path(listings_path).name}: {len(documents)} listing document(s)")
    return documents


# --------------------------------------------------------------------------
# Source 3: BNI training notes (data/BNI/*.txt) -- hierarchical indexing
# --------------------------------------------------------------------------

# Split on a sentence terminator followed by whitespace and a capital/quote.
_SENTENCE_RE = re.compile(r"(?<=[.!?])\s+(?=[\"'(\[]?[A-Z0-9])")

# Children shorter than this are dropped as headings/labels rather than embedded.
MIN_CHILD_CHARS = 20


def _split_sentences(paragraph):
    """Split a paragraph into child units.

    The notes are line-oriented -- headings and bullet lines often carry no
    terminal punctuation -- so split on newlines first, then on sentence
    boundaries within each line.
    """
    sentences = []
    for line in paragraph.splitlines():
        line = line.strip()
        if line:
            sentences.extend(s.strip() for s in _SENTENCE_RE.split(line) if s.strip())
    return sentences


def load_bni_hierarchical_nodes(txt_paths):
    """Build child (sentence) nodes that each carry their parent paragraph.

    Only the sentence text is embedded, so retrieval matches at sentence
    granularity. The full parent paragraph rides along in metadata as
    `parent_text`, which the query side feeds to the LLM as context.
    """
    nodes = []

    for txt_path in txt_paths:
        raw = txt_path.read_text(encoding="utf-8", errors="replace").strip()
        if not raw:
            print(f"  {txt_path.name}: skipped, file is empty")
            continue

        paragraphs = [p.strip() for p in re.split(r"\n\s*\n", raw) if p.strip()]
        file_sentences = 0

        for para_index, paragraph in enumerate(paragraphs):
            parent_id = f"{txt_path.stem}-p{para_index}"
            sentences = _split_sentences(paragraph)

            # Bare headings and labels ("Day 1", "Treasury") are too short to
            # match on usefully and only add retrieval noise; their paragraph
            # stays reachable through its longer sentences. If a paragraph is
            # nothing but short lines, keep it whole so it is still findable.
            long_enough = [s for s in sentences if len(s) >= MIN_CHILD_CHARS]
            sentences = long_enough or [paragraph]

            for sent_index, sentence in enumerate(sentences):
                node = TextNode(
                    text=sentence,
                    metadata={
                        "source": txt_path.name,
                        "doc_type": "bni_training",
                        "week": txt_path.stem,
                        "parent_id": parent_id,
                        "parent_index": para_index,
                        "sentence_index": sent_index,
                        "parent_text": paragraph,
                    },
                )
                # Embed the sentence alone -- the parent paragraph is context
                # for the LLM, not part of what the retriever matches against.
                node.excluded_embed_metadata_keys = list(node.metadata.keys())
                nodes.append(node)

            file_sentences += len(sentences)

        print(f"  {txt_path.name}: {len(paragraphs)} paragraph(s) -> {file_sentences} sentence node(s)")

    return nodes


# --------------------------------------------------------------------------
# Build the index
# --------------------------------------------------------------------------

# Hardcoding each source's path broke every time the data moved, and on Kaggle
# everything is flattened into one dataset dir. Instead, search a few plausible
# roots recursively and classify each file by name.
_kaggle_input = Path("/kaggle/input")
ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/revelelel/parsed-result"),
    Path("/kaggle/input/parsed-result"),
    *(sorted(_kaggle_input.glob("*")) if _kaggle_input.is_dir() else []),
    Path.cwd() / "data",
    Path.cwd(),
]

roots = []
for root in ROOT_CANDIDATES:
    if root.is_dir() and not any(root == seen or root in seen.parents for seen in roots):
        roots.append(root)

if not roots:
    raise FileNotFoundError("No data root found. Checked: "
                            + ", ".join(str(p) for p in ROOT_CANDIDATES))


def discover_sources(roots):
    """Classify every file under the roots into one of the three sources."""
    bni_files, listing_files, parsed_files = [], [], []
    seen_names = set()

    for root in roots:
        for path in sorted(root.rglob("*")):
            if not path.is_file() or path.stat().st_size == 0:
                continue

            # The same filename showing up under two roots is the same data
            # mounted twice, not two documents.
            if path.name in seen_names:
                continue

            if path.suffix.lower() == ".txt":
                bni_files.append(path)
            elif path.suffix.lower() == ".json":
                if path.stem.lower().startswith("laptop_listings"):
                    listing_files.append(path)
                else:
                    parsed_files.append(path)
            else:
                continue

            seen_names.add(path.name)

    # laptop_listings_filled.json supersedes the un-enriched laptop_listings.json.
    if any(p.stem.lower().endswith("_filled") for p in listing_files):
        listing_files = [p for p in listing_files if p.stem.lower().endswith("_filled")]

    return bni_files, listing_files, parsed_files


print("Searching for data under:", ", ".join(str(r) for r in roots))
bni_files, listing_files, parsed_files = discover_sources(roots)

all_nodes = []
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)

# 1. PDF-derived parsed results.
if parsed_files:
    print(f"\nParsed PDF results ({len(parsed_files)} file(s)):")
    parsed_docs = load_parsed_documents(parsed_files)
    parsed_nodes = splitter.get_nodes_from_documents(parsed_docs)
    all_nodes.extend(parsed_nodes)
    print(f"  -> {len(parsed_nodes)} chunk(s)")
else:
    print("\nNo parsed PDF JSON found.")

# 2. Laptop listings.
if listing_files:
    print(f"\nLaptop listings ({len(listing_files)} file(s)):")
    listing_docs = []
    for path in listing_files:
        listing_docs.extend(load_listing_documents(path))
    listing_nodes = splitter.get_nodes_from_documents(listing_docs)
    all_nodes.extend(listing_nodes)
    print(f"  -> {len(listing_nodes)} chunk(s)")
else:
    print("\nNo laptop listings JSON found.")

# 3. BNI training notes, hierarchically indexed (sentence child / paragraph parent).
if bni_files:
    print(f"\nBNI training notes ({len(bni_files)} file(s)):")
    bni_nodes = load_bni_hierarchical_nodes(bni_files)
    all_nodes.extend(bni_nodes)
    print(f"  -> {len(bni_nodes)} sentence chunk(s)")
else:
    print("\nNo BNI .txt notes found.")

if not all_nodes:
    raise RuntimeError("No documents found in any configured source.")

print(f"\nCreated {len(all_nodes)} embedding chunks across all sources.")

VectorStoreIndex(all_nodes, storage_context=storage_context)

print("Qdrant collections:", [item.name for item in client.get_collections().collections])
print(f"Done. Embeddings are stored in the {qdrant_collection} collection.")

## 4. Release the GPU

`HuggingFaceEmbedding` loaded bge-m3 into *this kernel's* VRAM and it stays
resident until dropped. vLLM sizes its KV cache against free memory at startup,
so anything still held here is memory the server never gets.

In [ ]:
import gc

import torch
from llama_index.core import Settings

Settings.embed_model = None
gc.collect()
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {free / 1e9:.1f} GB free of {total / 1e9:.1f} GB")

## 5. Start the local LLM server

vLLM runs as a **separate process**, not inside this kernel and not inside
`app.py`. Streamlit re-executes its whole script on every widget interaction,
so a model loaded there would be reloaded per session and would fight the
embedding model for VRAM. Here it loads once and answers over HTTP.

First run downloads ~10 GB, so allow several minutes.

In [ ]:
import os
import subprocess
import time

import requests

VLLM_PORT = 8000
MODEL = os.environ["HF_MODEL"]


def launch_vllm():
    # Re-running this cell leaves the old server holding the port and the VRAM.
    subprocess.run(["pkill", "-f", "vllm.entrypoints"], check=False)
    time.sleep(5)

    log = open("vllm.log", "w")
    proc = subprocess.Popen(
        [
            "python", "-m", "vllm.entrypoints.openai.api_server",
            "--model", MODEL,
            # app.py sends HF_MODEL as the model name and vLLM rejects a
            # mismatch, so the served name must be this exact string.
            "--served-model-name", MODEL,
            "--port", str(VLLM_PORT),
            # T4s are compute capability 7.5 and have no bfloat16.
            "--dtype", "half",
            "--tensor-parallel-size", "2",
            # Retrieved context plus history stays well under this, and a
            # smaller window leaves far more room for the KV cache on 16 GB cards.
            "--max-model-len", "8192",
            "--gpu-memory-utilization", "0.90",
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
    )

    for _ in range(1200):
        if proc.poll() is not None:
            raise RuntimeError("vLLM exited early:\n" + open("vllm.log").read()[-4000:])
        try:
            if requests.get(f"http://localhost:{VLLM_PORT}/v1/models", timeout=2).ok:
                break
        except requests.exceptions.RequestException:
            pass
        time.sleep(1)
    else:
        raise RuntimeError("vLLM never came up:\n" + open("vllm.log").read()[-4000:])

    print(f"vLLM is serving {MODEL} on http://localhost:{VLLM_PORT}/v1")
    return proc


vllm_proc = launch_vllm()

## 6. Smoke-test the server

Worth doing before Streamlit is in the picture: if this fails, the problem is
the model server, not the app.

In [ ]:
from openai import OpenAI

_client = OpenAI(base_url=os.environ["LOCAL_LLM_URL"], api_key="EMPTY")

print("Served models:", [m.id for m in _client.models.list().data])

_reply = _client.chat.completions.create(
    model=os.environ["HF_MODEL"],
    messages=[{"role": "user", "content": "Jawab dalam satu kalimat: apa itu RAG?"}],
    max_tokens=128,
    temperature=0.3,
)
print(_reply.choices[0].message.content)

## 7. Write `app.py`

Identical to the version in the repository — it carries no credentials of its
own and reads everything from the environment set in step 2, which the
Streamlit subprocess inherits.

In [ ]:
%%writefile app.py
"""
Streamlit chatbot UI: retrieves context from Qdrant (collection populated by
embedding.py) and generates answers with a Qwen2.5 instruct model.

Two LLM backends, selected with LLM_BACKEND:

  "local"  (default) -- an OpenAI-compatible server you run yourself, normally
           vLLM on the Kaggle GPU. Point LOCAL_LLM_URL at its /v1 endpoint.
           The weights never load into this process: Streamlit re-runs the
           script on every interaction, so a model held here would be reloaded
           and would compete with the embedding model for VRAM.
  "hf_api" -- the Hugging Face serverless Inference API. Needs HF_TOKEN with
           access to HF_MODEL.

Run with: streamlit run app.py
On Kaggle/Colab, start it from a notebook cell and open the ngrok tunnel there
(see kaggle_launcher.py) -- never from inside this file.
"""

import os

import qdrant_client
import streamlit as st
from dotenv import load_dotenv
from llama_index.core import Settings, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore

load_dotenv()

QDRANT_URL = os.environ.get("QDRANT_URL")
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY")
QDRANT_COLLECTION = os.environ.get("QDRANT_COLLECTION", "bni_training")
HF_TOKEN = os.environ.get("HF_TOKEN")
HF_MODEL = os.environ.get("HF_MODEL", "Qwen/Qwen2.5-14B-Instruct")

LLM_BACKEND = os.environ.get("LLM_BACKEND", "local").lower()
LOCAL_LLM_URL = os.environ.get("LOCAL_LLM_URL", "http://localhost:8000/v1")

# Must match embedding.py. Both read the same env var so the two cannot drift:
# a query embedded with a different model than the stored vectors still has the
# right dimensionality, so Qdrant accepts it and silently returns noise.
EMBED_MODEL_NAME = os.environ.get("EMBED_MODEL", "BAAI/bge-m3")

# bge-m3 embeds one short query per turn, which the CPU handles in well under a
# second. Defaulting it off the GPU leaves the whole card to vLLM, which sizes
# its KV cache once at startup and cannot give memory back afterwards.
EMBED_DEVICE = os.environ.get("EMBED_DEVICE", "cpu")

SYSTEM_PROMPT = (
    "You are a helpful assistant answering questions about BNI training "
    "notes, laptop procurement documents, and scraped laptop marketplace "
    "listings. Answer using only the provided context. If the answer "
    "isn't in the context, say you don't know instead of guessing. "
    "Reply in the same language as the question."
)

st.set_page_config(page_title="BNI Training Assistant", page_icon="💬")


@st.cache_resource(show_spinner="Loading embedding model and connecting to Qdrant...")
def get_index():
    Settings.embed_model = HuggingFaceEmbedding(
        model_name=EMBED_MODEL_NAME,
        device=EMBED_DEVICE,
    )
    client = qdrant_client.QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
    vector_store = QdrantVectorStore(client=client, collection_name=QDRANT_COLLECTION)
    return VectorStoreIndex.from_vector_store(vector_store)


@st.cache_resource
def get_llm_client():
    if LLM_BACKEND == "local":
        from openai import OpenAI

        # vLLM ignores the key but the client refuses to construct without one.
        return OpenAI(base_url=LOCAL_LLM_URL, api_key="EMPTY", timeout=600)

    if LLM_BACKEND == "hf_api":
        from huggingface_hub import InferenceClient

        return InferenceClient(model=HF_MODEL, token=HF_TOKEN)

    raise ValueError(f"LLM_BACKEND must be 'local' or 'hf_api', got {LLM_BACKEND!r}")


def stream_chat(client, messages, temperature):
    """Yield content deltas, hiding the two clients' differing entry points.

    Both backends emit OpenAI-shaped chunks, so only the call that opens the
    stream differs -- the delta handling below is shared.
    """
    if LLM_BACKEND == "local":
        stream = client.chat.completions.create(
            model=HF_MODEL,
            messages=messages,
            max_tokens=1024,
            temperature=temperature,
            stream=True,
        )
    else:
        stream = client.chat_completion(
            messages=messages,
            max_tokens=1024,
            temperature=temperature,
            stream=True,
        )

    for chunk in stream:
        choices = getattr(chunk, "choices", None) or []
        if choices:
            yield getattr(choices[0].delta, "content", None) or ""


def expand_to_parents(nodes):
    """Trade each retrieved child for the full parent it came from.

    embedding.py indexes the BNI notes hierarchically: a sentence is embedded
    as the child, with its whole paragraph carried in `parent_text`. Matching
    happens on the sentence, but the LLM should read the paragraph. Several
    sentences from one paragraph collapse into a single context chunk so the
    same text is not sent twice. Nodes from the other sources have no
    `parent_text` and are passed through as-is.
    """
    chunks = []
    seen_parents = set()

    for scored in nodes:
        node = scored.node
        parent_text = node.metadata.get("parent_text")

        if parent_text is None:
            chunks.append(node.get_content())
            continue

        parent_id = node.metadata.get("parent_id", parent_text)
        if parent_id in seen_parents:
            continue

        seen_parents.add(parent_id)
        chunks.append(parent_text)

    return chunks


def format_source_label(meta):
    source = meta.get("source", "?")

    if meta.get("doc_type") == "bni_training":
        return f"{source} — paragraph {meta.get('parent_index', '?')}"
    if meta.get("doc_type") == "laptop_listing":
        return f"{source} — {meta.get('brand', '?')} listing {meta.get('source_id', '?')}"
    if meta.get("doc_type") == "purchase_request":
        return f"{source} — request {meta.get('request_id', '?')}"
    if meta.get("page") is not None:
        return f"{source} — page {meta['page']}"
    return source


def build_user_turn(question, context_chunks):
    context = "\n\n---\n\n".join(context_chunks) if context_chunks else "(no matching context found)"
    return f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer using only the context above."


index = get_index()
llm_client = get_llm_client()

with st.sidebar:
    st.header("Settings")
    top_k = st.slider("Retrieved chunks", 1, 10, 5)
    temperature = st.slider("Temperature", 0.0, 1.0, 0.3, 0.05)
    if st.button("Clear chat"):
        st.session_state.messages = []
        st.rerun()

st.title("💬 BNI Training Assistant")
backend_label = "local vLLM" if LLM_BACKEND == "local" else "HF Inference API"
st.caption(f"RAG over Qdrant collection `{QDRANT_COLLECTION}` · {HF_MODEL} ({backend_label})")

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

question = st.chat_input("Ask something about the training documents...")

if question:
    st.session_state.messages.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.markdown(question)

    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(question)
    # Retrieval matched sentences; the LLM reads whole parent paragraphs.
    context_chunks = expand_to_parents(nodes)

    chat_messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    chat_messages.extend(
        {"role": m["role"], "content": m["content"]}
        for m in st.session_state.messages[:-1]
    )
    chat_messages.append({"role": "user", "content": build_user_turn(question, context_chunks)})

    with st.chat_message("assistant"):
        placeholder = st.empty()
        answer = ""
        try:
            for delta in stream_chat(llm_client, chat_messages, temperature):
                answer += delta
                if answer:
                    placeholder.markdown(answer + "▌")

            if not answer:
                raise RuntimeError(
                    "The model returned no text. Check that HF_MODEL names the "
                    "model the server actually loaded (vLLM rejects a mismatch)."
                )
            placeholder.markdown(answer)
        except Exception as e:
            answer = f"Error calling {HF_MODEL} via {backend_label}: {e}"
            placeholder.markdown(answer)

        if nodes:
            with st.expander("Sources"):
                for n in nodes:
                    meta = n.node.metadata
                    st.markdown(f"**{format_source_label(meta)}** (score: {n.score:.3f})")

                    parent_text = meta.get("parent_text")
                    if parent_text is not None:
                        st.caption("Matched sentence:")
                        st.text(n.node.get_content()[:500])
                        st.caption("Parent paragraph sent to the model:")
                        st.text(parent_text[:1500])
                    else:
                        st.text(n.node.get_content()[:500])

    st.session_state.messages.append({"role": "assistant", "content": answer})

## 8. Launch the UI

The tunnel is opened here rather than inside `app.py` for the same reason the
model is: this cell runs once, whereas `app.py` re-runs on every interaction
and would open a duplicate tunnel each time.

Interrupt the cell to stop the app.

In [ ]:
import os
import subprocess
import time

import requests
from pyngrok import ngrok

STREAMLIT_PORT = 8501


def launch_streamlit():
    # Re-running the cell leaves the old server holding the port, which is what
    # "Port 8501 is not available" means. Clear both sides before starting.
    ngrok.kill()
    subprocess.run(["pkill", "-f", "streamlit run"], check=False)
    time.sleep(2)

    log = open("streamlit.log", "w")
    proc = subprocess.Popen(
        [
            "streamlit", "run", "app.py",
            f"--server.port={STREAMLIT_PORT}",
            "--server.headless=true",
            "--server.address=0.0.0.0",
            "--server.enableCORS=false",
            "--server.enableXsrfProtection=false",
            "--browser.gatherUsageStats=false",
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
    )

    for _ in range(180):
        if proc.poll() is not None:
            raise RuntimeError("Streamlit exited early:\n" + open("streamlit.log").read())
        try:
            requests.get(f"http://localhost:{STREAMLIT_PORT}", timeout=2)
            break
        except requests.exceptions.RequestException:
            time.sleep(1)
    else:
        raise RuntimeError("Streamlit never came up:\n" + open("streamlit.log").read())

    ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
    url = ngrok.connect(addr=STREAMLIT_PORT, proto="http").public_url
    url = url.replace("http://", "https://", 1)
    print("Streamlit is up.")
    print("Public URL:", url)
    return proc, url


streamlit_proc, public_url = launch_streamlit()
# Keep the cell alive -- the tunnel dies when this process exits.
streamlit_proc.wait()

## 9. Shut down

Frees both ports and the GPU without restarting the session.

In [ ]:
import subprocess

subprocess.run(["pkill", "-f", "streamlit run"], check=False)
subprocess.run(["pkill", "-f", "vllm.entrypoints"], check=False)
ngrok.kill()
print("Stopped.")